# Fine-tuning Qwen2.5-0.5B-Instruct for Customer Support Conversations

This notebook fine-tunes a small instruction model using LoRA on cleaned conversation data.

In [ ]:
# Install dependencies
!pip install -q transformers peft accelerate datasets torch scikit-learn

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset


In [ ]:
# If using Google Drive, uncomment and mount:
# from google.colab import drive
# drive.mount('/content/drive')
# data_file = '/content/drive/My Drive/cleaned_conversations.jsonl'

# Otherwise, use the local path (relative to this notebook)
data_file = '/content/cleaned_conversations.jsonl'

data = []
with open(data_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} conversations")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Prepare dataset for instruction tuning
instructions = []
responses = []
for conv in data:
    turns = conv['turns']
    for i in range(0, len(turns)-1, 2):
        if i+1 < len(turns):
            instructions.append(turns[i]['text'])
            responses.append(turns[i+1]['text'])

# Limit to 50 examples (fast training on Colab)
instructions = instructions[:50]
responses = responses[:50]

print(f"Prepared {len(instructions)} instruction-response pairs")


In [ ]:
# Load model and tokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# Set up LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Tokenize and split
def tokenize_function(examples):
    texts = [f"Instruction: {instr}\nResponse: {resp}" for instr, resp in zip(examples['instruction'], examples['response'])]
    tokenized = tokenizer(texts, truncation=True, padding=True, max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()  # For causal LM loss computation
    return tokenized

dataset = Dataset.from_dict({'instruction': instructions, 'response': responses})
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Use Hugging Face dataset split (avoids sklearn issues)
split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()


In [ ]:
# Save weights and run a quick evaluation
# Save weights
model.save_pretrained("./results")

# Evaluation
results = trainer.evaluate()
print(results)

# Sample inference
input_text = "Instruction: Hello, I have an issue with my payment.\nResponse:"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_length=100)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Sample output:", response)
